In [2]:
# ================================================================
# STEP 1: CLEAN UNINSTALL EXISTING PACKAGES
# ================================================================

print("🧹 Cleaning existing packages...")

print("✅ Cleanup completed!")

# ================================================================
# STEP 2: SYSTEM DEPENDENCIES
# ================================================================

print("🔧 Installing system dependencies...")

!apt-get -qq update
!apt-get -qq install -y libfluidsynth1 libarchive-dev graphviz
!apt-get -qq install -y python3-dev build-essential

print("✅ System dependencies installed!")

# ================================================================
# STEP 3: INSTALL COMPATIBLE VERSIONS
# ================================================================

print("📦 Installing compatible Python packages...")

# Core scientific packages (stable versions)
!pip install numpy==1.24.3
!pip install pandas==2.0.3
!pip install scipy==1.10.1

# Scikit-learn and ML packages
!pip install scikit-learn==1.3.0
!pip install lightgbm==4.0.0
!pip install xgboost==1.7.6

# String similarity packages
!pip install fuzzywuzzy[speedup]==0.18.0
!pip install python-levenshtein==0.21.1
!pip install rapidfuzz==3.1.1
!pip install jellyfish==0.11.2
!pip install textdistance==4.6.0

# Utility packages
!pip install unidecode==1.3.6
!pip install optuna==3.2.0
!pip install tqdm==4.65.0

# Plotting (if needed)
!pip install matplotlib==3.7.1
!pip install seaborn==0.12.2

print("✅ All packages installed successfully!")

# ================================================================
# STEP 4: VERIFY INSTALLATION
# ================================================================

print("🔍 Verifying installation...")

try:
    import pandas as pd
    import numpy as np
    import sklearn
    import lightgbm as lgb
    import xgboost as xgb
    from fuzzywuzzy import fuzz
    import rapidfuzz
    import jellyfish
    import textdistance
    import optuna

    print("✅ All imports successful!")
    print(f"Pandas: {pd.__version__}")
    print(f"NumPy: {np.__version__}")
    print(f"Scikit-learn: {sklearn.__version__}")
    print(f"LightGBM: {lgb.__version__}")
    print(f"XGBoost: {xgb.__version__}")

except ImportError as e:
    print(f"❌ Import error: {e}")

print("🚀 Ready to run the advanced pipeline!")

🧹 Cleaning existing packages...
✅ Cleanup completed!
🔧 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Package 'libfluidsynth1' has no installation candidate
✅ System dependencies installed!
📦 Installing compatible Python packages...
  Using cached numpy-1.24.3.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a pr

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [1]:
import pandas as pd
import numpy as np
import re
import json
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModel, AutoConfig
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb

from fuzzywuzzy import fuzz, process
from rapidfuzz import fuzz as rapid_fuzz
import networkx as nx
from geopy.distance import geodesic
from geopy.geocoders import Nominatim

import optuna
from tqdm import tqdm
import gc


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ================================================================
# ADVANCED TURKISH ADDRESS NORMALIZER
# ================================================================

class AdvancedTurkishAddressNormalizer:
    """State-of-the-art Turkish address normalizer"""

    def __init__(self):
        # Comprehensive abbreviation mappings
        self.city_mappings = {
            'ist': 'istanbul', 'ank': 'ankara', 'izm': 'izmir', 'bur': 'bursa',
            'ada': 'adana', 'ant': 'antalya', 'mer': 'mersin', 'gaz': 'gaziantep',
            'kon': 'konya', 'kay': 'kayseri', 'esk': 'eskisehir', 'den': 'denizli',
            'sam': 'samsun', 'mal': 'malatya', 'ady': 'adiyaman', 'van': 'van',
            'tra': 'trabzon', 'erzr': 'erzurum', 'ela': 'elazig', 'die': 'diyarbakir',
            'san': 'sanliurfa', 'mar': 'mardin', 'hat': 'hatay', 'osk': 'osmaniye'
        }

        self.district_keywords = {
            'ilce', 'ilcesi', 'merkez', 'belediye', 'belediyesi'
        }

        self.neighborhood_keywords = {
            'mh', 'mah', 'mahalle', 'mahallesi', 'koyu', 'koy', 'kasaba', 'kasabasi',
            'belde', 'beldesi', 'semt', 'semti', 'bucak', 'bucagi'
        }

        self.street_mappings = {
            'cd': 'caddesi', 'cad': 'caddesi', 'cadde': 'caddesi',
            'sk': 'sokagi', 'sok': 'sokagi', 'sokak': 'sokagi',
            'blv': 'bulvari', 'bulv': 'bulvari', 'bulvar': 'bulvari',
            'st': 'sokagi', 'street': 'sokagi', 'str': 'sokagi',
            'ave': 'caddesi', 'avenue': 'caddesi',
            'km': 'kilometre', 'yol': 'yolu', 'osb': 'organize sanayi bolge'
        }

        self.building_mappings = {
            'apt': 'apartmani', 'apartman': 'apartmani', 'ap': 'apartmani',
            'sit': 'sitesi', 'site': 'sitesi', 'residence': 'sitesi',
            'plaza': 'plaza', 'plz': 'plaza', 'merkez': 'merkezi',
            'avm': 'alisveris merkezi', 'mall': 'alisveris merkezi',
            'is': 'is merkezi', 'tower': 'kulesi', 'kule': 'kulesi',
            'blk': 'blok', 'blok': 'blok', 'block': 'blok'
        }

        self.number_patterns = {
            'no_patterns': [
                r'no[:\s]*(\d+)(?:[/\-](\d+))?',
                r'numara[:\s]*(\d+)(?:[/\-](\d+))?',
                r'(\d+)(?:\s*numara)?(?:[/\-](\d+))?'
            ],
            'floor_patterns': [
                r'(\d+)\.?\s*kat',
                r'kat[:\s]*(\d+)',
                r'floor[:\s]*(\d+)',
                r'(\d+)\.?\s*kata'
            ],
            'apartment_patterns': [
                r'(\d+)\.?\s*daire',
                r'daire[:\s]*(\d+)',
                r'apartment[:\s]*(\d+)',
                r'flat[:\s]*(\d+)'
            ]
        }

        # Geographic hierarchy
        self.geographic_hierarchy = ['il', 'ilce', 'mahalle', 'sokak', 'bina', 'daire']

    def normalize_turkish_chars(self, text):
        """Enhanced Turkish character normalization"""
        if not isinstance(text, str):
            return ""

        # Turkish character mapping
        turkish_chars = {
            'ç': 'c', 'ğ': 'g', 'ı': 'i', 'ö': 'o', 'ş': 's', 'ü': 'u',
            'Ç': 'c', 'Ğ': 'g', 'İ': 'i', 'Ö': 'o', 'Ş': 's', 'Ü': 'u'
        }

        for tr_char, en_char in turkish_chars.items():
            text = text.replace(tr_char, en_char)

        return text.lower()

    def normalize_cities(self, text):
        """Normalize city abbreviations"""
        words = text.split()
        normalized_words = []

        for word in words:
            clean_word = re.sub(r'[^\w]', '', word.lower())
            if clean_word in self.city_mappings:
                normalized_words.append(self.city_mappings[clean_word])
            else:
                normalized_words.append(word)

        return ' '.join(normalized_words)

    def normalize_streets(self, text):
        """Advanced street type normalization"""
        # Handle compound street types
        text = re.sub(r'\b(sokak|sokagi)\s+(caddesi|cd|cad)\b', 'sokagi', text)
        text = re.sub(r'\b(cadde|caddesi)\s+(sokagi|sk|sok)\b', 'caddesi', text)

        # Standard street type replacement
        for abbrev, full_form in self.street_mappings.items():
            pattern = r'\b' + re.escape(abbrev) + r'\b'
            text = re.sub(pattern, full_form, text, flags=re.IGNORECASE)

        return text

    def normalize_buildings(self, text):
        """Normalize building types"""
        for abbrev, full_form in self.building_mappings.items():
            pattern = r'\b' + re.escape(abbrev) + r'\b'
            text = re.sub(pattern, full_form, text, flags=re.IGNORECASE)

        return text

    def standardize_numbers(self, text):
        """Advanced number standardization"""
        # Building number patterns
        text = re.sub(r'(\d+)[/\-](\d+)(?:\s*(?:daire|dr|d))?', r'numara \1 daire \2', text)
        text = re.sub(r'no[:\s=]*(\d+)', r'numara \1', text, flags=re.IGNORECASE)

        # Floor patterns
        text = re.sub(r'(\d+)\.?\s*(?:kat|kt|k)(?:\s|$)', r'\1 kat ', text, flags=re.IGNORECASE)

        # Apartment patterns
        text = re.sub(r'(\d+)\.?\s*(?:daire|dr|d)(?:\s|$)', r'\1 daire ', text, flags=re.IGNORECASE)

        # Block patterns
        text = re.sub(r'([a-zA-Z]?\d*)\.?\s*(?:blok|blk|b)(?:\s|$)', r'\1 blok ', text, flags=re.IGNORECASE)

        return text

    def extract_hierarchical_components(self, text):
        """Extract hierarchical address components"""
        components = {
            'il': '',
            'ilce': '',
            'mahalle': '',
            'sokak': '',
            'numara': '',
            'kat': '',
            'daire': '',
            'blok': '',
            'posta_kodu': ''
        }

        # Extract postal code
        postal_match = re.search(r'\b(\d{5})\b', text)
        if postal_match:
            components['posta_kodu'] = postal_match.group(1)

        # Extract numbers
        for pattern_type, patterns in self.number_patterns.items():
            for pattern in patterns:
                match = re.search(pattern, text, re.IGNORECASE)
                if match:
                    if 'no_' in pattern_type:
                        components['numara'] = match.group(1)
                        if match.group(2):
                            components['daire'] = match.group(2)
                    elif 'floor' in pattern_type:
                        components['kat'] = match.group(1)
                    elif 'apartment' in pattern_type:
                        components['daire'] = match.group(1)
                    break

        # Extract mahalle/neighborhood
        for keyword in self.neighborhood_keywords:
            pattern = r'(\w+)\s+' + keyword
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                components['mahalle'] = match.group(1)
                break

        return components

    def normalize(self, address):
        """Complete normalization pipeline"""
        if pd.isna(address) or not isinstance(address, str):
            return ""

        # Step 1: Basic cleaning
        address = address.strip().lower()

        # Step 2: Turkish character normalization
        address = self.normalize_turkish_chars(address)

        # Step 3: City normalization
        address = self.normalize_cities(address)

        # Step 4: Street normalization
        address = self.normalize_streets(address)

        # Step 5: Building normalization
        address = self.normalize_buildings(address)

        # Step 6: Number standardization
        address = self.standardize_numbers(address)

        # Step 7: Clean punctuation and extra spaces
        address = re.sub(r'[^\w\s]', ' ', address)
        address = re.sub(r'\s+', ' ', address)

        return address.strip()


In [ ]:
# ================================================================
# TURKISH BERT MODEL FOR ADDRESS EMBEDDINGS
# ================================================================

class TurkishAddressBERT:
    """Fine-tuned Turkish BERT for address embeddings"""

    def __init__(self, model_name='dbmdz/bert-base-turkish-cased'):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()

    def encode_batch(self, texts, batch_size=32, max_length=128):
        """Encode texts in batches"""
        embeddings = []

        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding with Turkish BERT"):
            batch_texts = texts[i:i+batch_size]

            # Tokenize
            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            ).to(device)

            # Get embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use [CLS] token embedding
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(batch_embeddings)

        return np.vstack(embeddings)

In [ ]:
# ================================================================
# ADVANCED SIMILARITY CALCULATOR
# ================================================================

class AdvancedSimilarityCalculator:
    """Multiple similarity metrics for address matching"""

    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 6),  # Extended n-gram range
            max_features=100000,
            lowercase=True
        )

    def calculate_fuzzy_similarities(self, addr1_list, addr2_list):
        """Calculate multiple fuzzy similarities"""
        similarities = []

        for addr1, addr2 in zip(addr1_list, addr2_list):
            sim_dict = {
                'ratio': fuzz.ratio(addr1, addr2) / 100.0,
                'partial_ratio': fuzz.partial_ratio(addr1, addr2) / 100.0,
                'token_sort': fuzz.token_sort_ratio(addr1, addr2) / 100.0,
                'token_set': fuzz.token_set_ratio(addr1, addr2) / 100.0,
                'rapid_ratio': rapid_fuzz.ratio(addr1, addr2) / 100.0
            }
            similarities.append(sim_dict)

        return pd.DataFrame(similarities)

    def calculate_tfidf_similarity(self, train_addresses, test_addresses):
        """Enhanced TF-IDF similarity"""
        all_addresses = train_addresses + test_addresses
        tfidf_matrix = self.tfidf_vectorizer.fit_transform(all_addresses)

        train_matrix = tfidf_matrix[:len(train_addresses)]
        test_matrix = tfidf_matrix[len(train_addresses):]

        similarity_matrix = cosine_similarity(test_matrix, train_matrix)
        return similarity_matrix

    def calculate_hierarchical_similarity(self, components1_list, components2_list):
        """Hierarchical component-based similarity"""
        similarities = []
        weights = {
            'il': 0.15,
            'ilce': 0.15,
            'mahalle': 0.25,
            'sokak': 0.25,
            'numara': 0.15,
            'daire': 0.05
        }

        for comp1, comp2 in zip(components1_list, components2_list):
            total_sim = 0
            total_weight = 0

            for component in weights:
                if comp1.get(component) and comp2.get(component):
                    sim = fuzz.ratio(comp1[component], comp2[component]) / 100.0
                    total_sim += sim * weights[component]
                    total_weight += weights[component]

            final_sim = total_sim / total_weight if total_weight > 0 else 0
            similarities.append(final_sim)

        return np.array(similarities)

In [ ]:
# ================================================================
# GEOSPATIAL EMBEDDING MODEL
# ================================================================

class GeospatialEmbedding:
    """Geospatial features for address matching"""

    def __init__(self):
        self.geolocator = Nominatim(user_agent="address_matcher")
        self.coordinate_cache = {}

    def get_coordinates(self, address, use_cache=True):
        """Get coordinates for address with caching"""
        if use_cache and address in self.coordinate_cache:
            return self.coordinate_cache[address]

        try:
            location = self.geolocator.geocode(address, timeout=5)
            if location:
                coords = (location.latitude, location.longitude)
                if use_cache:
                    self.coordinate_cache[address] = coords
                return coords
        except:
            pass

        return None

    def calculate_geographic_distance(self, addr1_list, addr2_list):
        """Calculate geographic distances between addresses"""
        distances = []

        for addr1, addr2 in zip(addr1_list, addr2_list):
            coords1 = self.get_coordinates(addr1)
            coords2 = self.get_coordinates(addr2)

            if coords1 and coords2:
                distance = geodesic(coords1, coords2).kilometers
                # Normalize distance (closer = higher similarity)
                similarity = 1 / (1 + distance)
            else:
                similarity = 0.5  # Neutral similarity if coords unavailable

            distances.append(similarity)

        return np.array(distances)

In [ ]:
# ================================================================
# GRAPH NEURAL NETWORK FOR HIERARCHICAL ADDRESSES
# ================================================================

class AddressGraphNN(nn.Module):
    """Graph Neural Network for address hierarchy"""

    def __init__(self, input_dim=768, hidden_dim=256, output_dim=128):
        super(AddressGraphNN, self).__init__()
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.linear3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.2)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.dropout(x)
        x = self.activation(self.linear2(x))
        x = self.dropout(x)
        x = self.linear3(x)
        return x


NameError: name 'nn' is not defined

In [ ]:
# ================================================================
# ENSEMBLE MODEL WITH OPTIMIZATION
# ================================================================

class OptimizedEnsemble:
    """Optimized ensemble using Optuna"""

    def __init__(self):
        self.models = {}
        self.weights = None
        self.best_weights = None

    def objective(self, trial, X_val, y_val):
        """Optuna objective function"""
        # Suggest weights for different models
        w1 = trial.suggest_float('tfidf_weight', 0.1, 0.4)
        w2 = trial.suggest_float('bert_weight', 0.2, 0.6)
        w3 = trial.suggest_float('fuzzy_weight', 0.1, 0.3)
        w4 = trial.suggest_float('geo_weight', 0.0, 0.2)

        # Normalize weights
        total_weight = w1 + w2 + w3 + w4
        weights = [w1/total_weight, w2/total_weight, w3/total_weight, w4/total_weight]

        # Calculate ensemble predictions
        ensemble_scores = np.zeros(len(X_val))
        for i, weight in enumerate(weights):
            if i < X_val.shape[1]:
                ensemble_scores += weight * X_val[:, i]

        # Calculate accuracy (assuming binary classification for similarity)
        predictions = ensemble_scores > 0.5
        accuracy = (predictions == y_val).mean()

        return accuracy

    def optimize_weights(self, X_val, y_val, n_trials=100):
        """Optimize ensemble weights using Optuna"""
        study = optuna.create_study(direction='maximize')
        study.optimize(
            lambda trial: self.objective(trial, X_val, y_val),
            n_trials=n_trials
        )

        self.best_weights = study.best_params
        return self.best_weights


In [ ]:
# ================================================================
# MAIN PIPELINE INTEGRATION
# ================================================================

class AdvancedAddressMatchingPipeline:
    """Complete advanced pipeline"""

    def __init__(self):
        self.normalizer = AdvancedTurkishAddressNormalizer()
        self.bert_model = TurkishAddressBERT()
        self.similarity_calc = AdvancedSimilarityCalculator()
        self.geo_embedding = GeospatialEmbedding()
        self.ensemble = OptimizedEnsemble()

        # Sentence transformers for comparison
        self.sentence_transformer = SentenceTransformer('emrecan/bert-base-turkish-cased-mean-nli-stsb-tr')
        self.xlm_model = SentenceTransformer('sentence-transformers/paraphrase-xlm-r-multilingual-v1')

    def preprocess_data(self, train_df, test_df):
        """Advanced preprocessing pipeline"""
        print("🔧 Starting advanced preprocessing...")

        # Normalize addresses
        train_df['normalized_address'] = train_df['address'].apply(self.normalizer.normalize)
        test_df['normalized_address'] = test_df['address'].apply(self.normalizer.normalize)

        # Extract hierarchical components
        print("📍 Extracting hierarchical components...")
        train_df['components'] = train_df['normalized_address'].apply(
            self.normalizer.extract_hierarchical_components
        )
        test_df['components'] = test_df['normalized_address'].apply(
            self.normalizer.extract_hierarchical_components
        )

        # Create structured representations
        train_df['structured_address'] = train_df.apply(
            lambda row: self.create_structured_representation(row['components']), axis=1
        )
        test_df['structured_address'] = test_df.apply(
            lambda row: self.create_structured_representation(row['components']), axis=1
        )

        return train_df, test_df

    def create_structured_representation(self, components):
        """Create structured address representation"""
        structure = []
        hierarchy = ['il', 'ilce', 'mahalle', 'sokak', 'numara', 'kat', 'daire']

        for level in hierarchy:
            if components.get(level):
                structure.append(f"{level}:{components[level]}")

        return ' '.join(structure)

    def generate_embeddings(self, addresses, model_type='bert'):
        """Generate embeddings using specified model"""
        if model_type == 'bert':
            return self.bert_model.encode_batch(addresses)
        elif model_type == 'sentence_transformer':
            return self.sentence_transformer.encode(addresses, show_progress_bar=True)
        elif model_type == 'xlm':
            return self.xlm_model.encode(addresses, show_progress_bar=True)
        else:
            raise ValueError(f"Unknown model type: {model_type}")

    def create_features(self, train_df, test_df):
        """Create comprehensive feature set"""
        print("🎯 Creating comprehensive features...")

        features = {}

        # 1. BERT embeddings
        print("📝 Generating BERT embeddings...")
        train_bert = self.generate_embeddings(train_df['normalized_address'].tolist(), 'bert')
        test_bert = self.generate_embeddings(test_df['normalized_address'].tolist(), 'bert')

        # 2. Sentence transformer embeddings
        print("🔤 Generating Sentence Transformer embeddings...")
        train_sent = self.generate_embeddings(train_df['normalized_address'].tolist(), 'sentence_transformer')
        test_sent = self.generate_embeddings(test_df['normalized_address'].tolist(), 'sentence_transformer')

        # 3. XLM-R embeddings
        print("🌐 Generating XLM-R embeddings...")
        train_xlm = self.generate_embeddings(train_df['normalized_address'].tolist(), 'xlm')
        test_xlm = self.generate_embeddings(test_df['normalized_address'].tolist(), 'xlm')

        features['train_embeddings'] = {
            'bert': train_bert,
            'sentence_transformer': train_sent,
            'xlm': train_xlm
        }

        features['test_embeddings'] = {
            'bert': test_bert,
            'sentence_transformer': test_sent,
            'xlm': test_xlm
        }

        return features

    def predict_with_ensemble(self, features, train_labels):
        """Advanced ensemble prediction"""
        print("🚀 Making ensemble predictions...")

        # Encode labels
        label_encoder = LabelEncoder()
        encoded_labels = label_encoder.fit_transform(train_labels)

        predictions = {}

        # Predict with each model
        for model_name, train_emb in features['train_embeddings'].items():
            print(f"   Predicting with {model_name}...")
            test_emb = features['test_embeddings'][model_name]

            # Calculate similarities
            similarities = cosine_similarity(test_emb, train_emb)

            # Find best matches
            best_matches = similarities.argmax(axis=1)
            model_predictions = label_encoder.inverse_transform(best_matches)
            predictions[model_name] = model_predictions

            # Store confidence scores
            predictions[f'{model_name}_confidence'] = similarities.max(axis=1)

        # Weighted ensemble (you can optimize these weights)
        weights = {
            'bert': 0.4,
            'sentence_transformer': 0.35,
            'xlm': 0.25
        }

        # Create final ensemble prediction
        final_predictions = []

        for i in range(len(predictions['bert'])):
            label_scores = {}

            for model_name, weight in weights.items():
                pred_label = predictions[model_name][i]
                confidence = predictions[f'{model_name}_confidence'][i]

                if pred_label not in label_scores:
                    label_scores[pred_label] = 0
                label_scores[pred_label] += weight * confidence

            # Choose label with highest weighted score
            best_label = max(label_scores.items(), key=lambda x: x[1])[0]
            final_predictions.append(best_label)

        return final_predictions, predictions

In [ ]:
# ================================================================
# USAGE EXAMPLE
# ================================================================

def run_advanced_pipeline():
    """Run the complete advanced pipeline"""

    # Initialize pipeline
    pipeline = AdvancedAddressMatchingPipeline()

    # Load data
    print("📊 Loading data...")
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")

    # Preprocess data
    train_df, test_df = pipeline.preprocess_data(train_df, test_df)

    # Create features
    features = pipeline.create_features(train_df, test_df)

    # Make predictions
    final_predictions, all_predictions = pipeline.predict_with_ensemble(
        features, train_df['label']
    )

    # Create submission
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'label': final_predictions
    })

    submission_df.to_csv('advanced_submission.csv', index=False)
    print("✅ Advanced submission saved!")

    # Save individual model predictions for analysis
    for model_name in ['bert', 'sentence_transformer', 'xlm']:
        individual_submission = pd.DataFrame({
            'id': test_df['id'],
            'label': all_predictions[model_name]
        })
        individual_submission.to_csv(f'{model_name}_submission.csv', index=False)

    print("🎯 All submissions generated!")
    return submission_df

# Run the pipeline
if __name__ == "__main__":
    # Clear memory
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # Run advanced pipeline
    result_df = run_advanced_pipeline()

    print("🏆 Advanced pipeline completed!")
    print("Generated submissions:")
    print("- advanced_submission.csv (ensemble)")
    print("- bert_submission.csv (Turkish BERT)")
    print("- sentence_transformer_submission.csv (Turkish Sentence BERT)")
    print("- xlm_submission.csv (XLM-R multilingual)")